# 🤖 Complete Learning Guide: Building AI-Based Semantic Search Solutions

## Welcome, Beginners! 👋

This notebook will teach you how to build intelligent search systems using **embeddings** and **vector databases**. We'll use real examples from the **Stock Analyzer** project.

### What You'll Learn:
1. ✅ What **embeddings** are and why they're powerful
2. ✅ How to convert text to **meaningful vectors**
3. ✅ How to **chunk large documents** for better search
4. ✅ How to use **FAISS** for fast similarity search
5. ✅ How to build **Q&A systems** on top of embeddings
6. ✅ How to **manage and organize** embeddings at scale

### Real-World Application:
The **Stock Analyzer** system uses these techniques to:
- 📊 Analyze stock reports
- 🔍 Answer questions about stocks
- 📈 Compare multiple companies intelligently
- ⚡ Return results in milliseconds using vector search

Let's dive in! 🚀

---

## Section 1️⃣: Understanding Embeddings and Semantic Search

### What is an Embedding?

An **embedding** is a way to represent text (or any information) as a **list of numbers**. Think of it like this:

```
TEXT: "ASIANPAINT has strong ROE"
         ↓↓↓ (Convert with ML model)
EMBEDDING: [0.123, -0.456, 0.789, ..., 0.234]  ← 384 numbers!
```

### Why Embeddings Matter?

**Keyword Search (Old Way):**
```
Query: "What is ROE?"
Results: Only finds documents with "ROE" (exact match)
❌ Problem: What about "return on equity"? Won't find it!
```

**Semantic Search (New Way with Embeddings):**
```
Query: "What is ROE?"
         ↓ Convert to embedding
Compare with all documents' embeddings
Find documents with SIMILAR meaning
Results: 
  - "return on equity is 15%" ✅
  - "ROE is 15%" ✅
  - "the stock earned 15% on equity" ✅
  
❌ Won't match: "the team returned early" (different meaning)
```

### Key Insight
Embeddings capture **semantic meaning**. Similar ideas have similar embeddings, even if words are different.

---

## Section 2️⃣: Setting Up the Embedding Model

### The Core Function: `get_embedding_model()`

This function tries multiple embedding backends with **smart fallbacks**. Here's the flow:

```
Try #1: langchain-huggingface 
   ↓ (if fails)
Try #2: sentence-transformers directly
   ↓ (if fails)
Try #3: langchain-community embeddings
   ↓ (if fails)
Try #4: Hash-based embeddings (always works!)
```

This pattern ensures the system **never crashes** - it has 4 fallback options!

### The Embedding Model We Use

**Model Name:** `sentence-transformers/all-MiniLM-L6-v2`

| Property | Value | Why? |
|----------|-------|------|
| Output Size | 384 dimensions | Good balance of speed vs accuracy |
| Speed | ⚡ Very fast | Lightweight model |
| Accuracy | 📊 Good | Understands semantic meaning well |
| Size | 27MB | Fits in memory easily |
| No API Call | ✅ Local model | No external API needed |

In [1]:
# Example: Setting up the Embedding Model

import sys
import numpy as np
from pathlib import Path

# Add project to path
sys.path.insert(0, '/Users/macos/projects/personal/stock_annalyzer')

# Import the function from actual code
from src.embeddings.analysis_embedding_store import get_embedding_model

# Get the embedding model (from real code)
print("Loading embedding model...")
embeddings_model = get_embedding_model()

print(f"✅ Model loaded: {type(embeddings_model).__name__}")
print(f"\nModel details:")
print(f"  - Can embed queries: {hasattr(embeddings_model, 'embed_query')}")
print(f"  - Can embed documents: {hasattr(embeddings_model, 'embed_documents')}")

Loading embedding model...
✅ Model loaded: HuggingFaceEmbeddings

Model details:
  - Can embed queries: True
  - Can embed documents: True


In [2]:
# Example: Creating Embeddings from Text

sample_texts = [
    "ASIANPAINT has strong ROE and ROCE metrics",
    "Return on equity is 15% for this company",
    "The paint company is valued at 50 P/E ratio",
    "What is the current stock price?"
]

print("Creating embeddings for sample texts...\n")

for text in sample_texts:
    embedding = embeddings_model.embed_query(text)
    
    print(f"Text: {text}")
    print(f"  Embedding dimension: {len(embedding)}")
    print(f"  First 5 values: {[round(v, 3) for v in embedding[:5]]}")
    print(f"  Last 5 values: {[round(v, 3) for v in embedding[-5:]]}")
    print()

Creating embeddings for sample texts...

Text: ASIANPAINT has strong ROE and ROCE metrics
  Embedding dimension: 384
  First 5 values: [0.016, -0.02, -0.016, 0.023, -0.016]
  Last 5 values: [0.016, 0.067, -0.071, 0.018, -0.005]

Text: Return on equity is 15% for this company
  Embedding dimension: 384
  First 5 values: [0.014, 0.005, -0.017, -0.018, -0.038]
  Last 5 values: [0.006, -0.04, -0.092, 0.005, -0.028]

Text: The paint company is valued at 50 P/E ratio
  Embedding dimension: 384
  First 5 values: [-0.009, 0.051, -0.015, -0.034, 0.041]
  Last 5 values: [-0.034, -0.003, -0.074, -0.025, 0.022]

Text: What is the current stock price?
  Embedding dimension: 384
  First 5 values: [0.0, -0.038, -0.021, 0.023, -0.033]
  Last 5 values: [-0.032, 0.029, -0.156, 0.002, 0.093]



In [3]:
# Example: Semantic Similarity Comparison

# Let's show why semantic search is better than keyword search
def calculate_similarity(emb1, emb2):
    """Calculate cosine similarity between two embeddings"""
    dot_product = np.dot(emb1, emb2)
    norm1 = np.linalg.norm(emb1)
    norm2 = np.linalg.norm(emb2)
    return dot_product / (norm1 * norm2)

# Query and documents
query = "What is ROE?"
documents = [
    "ROE is 15%",  # Exact match - should be highest
    "Return on equity is 15%",  # Same meaning, different words
    "The stock price is 500",  # Unrelated
    "Dividend yield is 2%"  # Related but different metric
]

# Get embeddings
query_emb = embeddings_model.embed_query(query)
doc_embs = [embeddings_model.embed_query(doc) for doc in documents]

print(f"Query: '{query}'")
print("\nSimilarity Scores (0 = unrelated, 1 = identical):\n")

scores = []
for doc, doc_emb in zip(documents, doc_embs):
    similarity = calculate_similarity(query_emb, doc_emb)
    scores.append((doc, similarity))
    print(f"  '{doc}'")
    print(f"    Similarity: {similarity:.4f}")

# Sort by similarity
scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)
print("\n📊 Ranked by Relevance:")
for i, (doc, score) in enumerate(scores_sorted, 1):
    print(f"  {i}. [{score:.4f}] {doc}")

Query: 'What is ROE?'

Similarity Scores (0 = unrelated, 1 = identical):

  'ROE is 15%'
    Similarity: 0.7192
  'Return on equity is 15%'
    Similarity: 0.1952
  'The stock price is 500'
    Similarity: 0.1282
  'Dividend yield is 2%'
    Similarity: 0.0900

📊 Ranked by Relevance:
  1. [0.7192] ROE is 15%
  2. [0.1952] Return on equity is 15%
  3. [0.1282] The stock price is 500
  4. [0.0900] Dividend yield is 2%


---

## Section 3️⃣: Chunking Text for Better Search Results

### Why Chunking?

When you have a **long document** (like a 10,000-word stock analysis), you can't just embed the whole thing:

```
❌ Problem: Embedding entire 10KB analysis
  - Loses detailed information
  - Q&A becomes vague
  - Harder to match specific queries

✅ Solution: Split into small chunks (500 chars each)
  - Each chunk has focused topic
  - Specific questions match specific chunks
  - Better Q&A accuracy
```

### Chunking Strategy in Stock Analyzer

The `chunk_analysis()` method:

1. **Takes a full analysis dict** with 7 sections:
   - company_overview
   - quantitative_analysis
   - qualitative_analysis
   - shareholding_analysis
   - investment_thesis
   - valuation_recommendation
   - conclusion

2. **Converts HTML to plain text** (removes formatting noise)

3. **Splits by overlapping chunks**:
   - Chunk size: 500 characters
   - Overlap: 100 characters (to preserve context between chunks)
   - Example: Chunk 1 = chars 0-500, Chunk 2 = chars 400-900

4. **Stores metadata**:
   ```python
   (chunk_text, section_name, stock_symbol)
   ```

In [5]:
# Example: Chunking Analysis Text

from src.embeddings.analysis_embedding_store import AnalysisEmbeddingStore

# Create a mock analysis result (like what we'd get from LLM)
sample_analysis = {
    'company_overview': '''
        <p>Asian Paints is India's leading paint company with a strong market presence.</p>
        <p>Founded in 1942, the company has grown to become a household name in the paint industry.</p>
        <p>The company operates across residential, commercial, and industrial segments.</p>
        <p>With over 1000 paint shades, Asian Paints dominates the Indian paint market with 34% market share.</p>
    ''',
    'quantitative_analysis': '''
        <p>The company has demonstrated consistent financial performance over the past decade.</p>
        <p>Revenue CAGR: 15%, Profit CAGR: 20%, ROE: 28%, ROCE: 32%</p>
        <p>Current P/E ratio is 45, dividend yield is 1.2%, Book value per share is Rs 850</p>
        <p>EPS has grown from Rs 50 to Rs 150 in the last 5 years</p>
    ''',
    'qualitative_analysis': '''No content for this example''',
    'shareholding_analysis': '''No content for this example''',
    'investment_thesis': '''No content for this example''',
    'valuation_recommendation': '''No content for this example''',
    'conclusion': '''No content for this example'''
}

# Initialize the store
store = AnalysisEmbeddingStore()

# Chunk the analysis
chunks = store.chunk_analysis(sample_analysis, 'ASIANPAINT', chunk_size=500)

print(f"✅ Created {len(chunks)} chunks from analysis\n")
print("=" * 80)

for i, (chunk_text, section, symbol) in enumerate(chunks, 1):
    print(f"\nChunk {i}:")
    print(f"  Section: {section}")
    print(f"  Symbol: {symbol}")
    print(f"  Length: {len(chunk_text)} chars")
    print(f"  Content: {chunk_text[:100]}...")
    print("-" * 80)

✅ Created 2 chunks from analysis


Chunk 1:
  Section: company_overview
  Symbol: ASIANPAINT
  Length: 342 chars
  Content: Asian Paints is India's leading paint company with a strong market presence. Founded in 1942, the co...
--------------------------------------------------------------------------------

Chunk 2:
  Section: quantitative_analysis
  Symbol: ASIANPAINT
  Length: 275 chars
  Content: The company has demonstrated consistent financial performance over the past decade. Revenue CAGR: 15...
--------------------------------------------------------------------------------


---

## Section 4️⃣: Creating and Storing FAISS Indexes

### What is FAISS?

**FAISS** = Facebook AI Similarity Search

It's a **super-fast library** for searching through millions of embeddings. Instead of comparing your query embedding with each document manually:

```
❌ Naive approach (slow):
  Compare query with 1000 documents = 1000 comparisons
  
✅ FAISS (fast):
  Use special data structure (index)
  Compare query once = result instantly
  Works with millions of embeddings!
```

### How FAISS Works in Our System

```
Step 1: Create chunks from analysis
         ↓
Step 2: Convert chunks to embeddings (384-dim vectors)
         ↓
Step 3: Build FAISS index (IndexFlatL2)
         ↓
Step 4: Save index to disk (faiss_index.bin)
         ↓
Step 5: Save metadata to disk (metadata.pkl)
```

### Index Type: IndexFlatL2

| Property | Meaning |
|----------|---------|
| **Flat** | No compression (exact distances) |
| **L2** | Euclidean distance (most common) |
| **Distance** | Measures how different two vectors are |

**Formula:** Distance = √[(x1-y1)² + (x2-y2)² + ... + (x384-y384)²]

- **Small distance** = similar documents ✅
- **Large distance** = different documents ❌

In [7]:
# Example: Saving Analysis Embeddings with FAISS

import tempfile
import os

# Create a temporary directory for this example
temp_storage = tempfile.mkdtemp(prefix="stock_analyzer_demo_")

print(f"Using temporary storage: {temp_storage}\n")

# Initialize store with custom storage path
store_with_custom_path = AnalysisEmbeddingStore(storage_path=Path(temp_storage))

# Prepare analysis data for ASIANPAINT
analysis_data = {
    'company_overview': '''
        <p>Asian Paints Limited is India's largest paint company.</p>
        <p>The company has strong presence in residential, commercial and industrial paints.</p>
        <p>Founded in 1942, Asian Paints has grown to have 34% market share in India.</p>
        <p>The company distributes through 5000+ dealers across India.</p>
        <p>Quality and brand trust are the key pillars of Asian Paints strategy.</p>
    ''',
    'quantitative_analysis': '''
        <p>Financial Performance: Strong revenue growth of 15% CAGR over last 5 years.</p>
        <p>Profitability: Net profit margin of 12-15%, consistently profitable.</p>
        <p>Returns: ROE of 28%, ROCE of 32%, indicating efficient capital deployment.</p>
        <p>Valuation: P/E ratio of 45x, trading above historical average.</p>
        <p>Dividend: Annual dividend yield of 1.2%, paid consistently for 20+ years.</p>
    ''',
    'qualitative_analysis': '''
        <p>Brand Strength: Asian Paints is a household name with high brand recall.</p>
        <p>Distribution: Strong distribution network with 5000+ dealers.</p>
        <p>Innovation: Continuous innovation in paint products and formulations.</p>
        <p>Competition: Main competitors are Berger Paints and Nippon Paint.</p>
    ''',
    'shareholding_analysis': '''
        <p>Promoter holding: 50% controlled by Akzo Nobel N.V.</p>
        <p>Institutional investors: 30% holding by FII and DII.</p>
        <p>Public holding: 20% held by public shareholders.</p>
    ''',
    'investment_thesis': '''
        <p>Investment case: Strong brand, market leader, consistent growth.</p>
        <p>Key triggers: Margin improvement, geographic expansion, new product launches.</p>
    ''',
    'valuation_recommendation': '''
        <p>Fair value: Rs 3000-3200 based on DCF analysis.</p>
        <p>Recommendation: BUY for long-term investors.</p>
    ''',
    'conclusion': '''
        <p>Asian Paints presents a strong investment opportunity with stable growth.</p>
        <p>Quality business at a reasonable valuation for long-term wealth creation.</p>
    '''
}

# Save embeddings to FAISS
print("Saving embeddings to FAISS...\n")
success, message = store_with_custom_path.save_analysis_embeddings(
    symbol='ASIANPAINT',
    analysis_result=analysis_data,
    overwrite=True
)

print(f"Result: {success}")
print(f"Message: {message}\n")

# Show what was saved
if success:
    asianpaint_dir = Path(temp_storage) / 'ASIANPAINT'
    print(f"✅ Files saved in: {asianpaint_dir}\n")
    
    if asianpaint_dir.exists():
        files = list(asianpaint_dir.iterdir())
        for file in files:
            size_kb = file.stat().st_size / 1024
            print(f"  📄 {file.name} ({size_kb:.1f} KB)")

Using temporary storage: /var/folders/q6/3xcrqbk137b44f98n3dhhtw40000gn/T/stock_analyzer_demo_cv7qdcpt

Saving embeddings to FAISS...

Result: True
Message: Saved 7 embeddings for ASIANPAINT to /var/folders/q6/3xcrqbk137b44f98n3dhhtw40000gn/T/stock_analyzer_demo_cv7qdcpt/ASIANPAINT/faiss_index.bin

✅ Files saved in: /var/folders/q6/3xcrqbk137b44f98n3dhhtw40000gn/T/stock_analyzer_demo_cv7qdcpt/ASIANPAINT

  📄 metadata.pkl (1.8 KB)
  📄 faiss_index.bin (10.5 KB)


---

## Section 5️⃣: Searching Through Embeddings

### The Search Process

```
Step 1: User asks a question
        "What is the ROE of ASIANPAINT?"
        
Step 2: Convert question to embedding (384-dim vector)
        embeddings_model.embed_query(question)
        
Step 3: Load FAISS index from disk
        Load faiss_index.bin for ASIANPAINT
        
Step 4: Find similar chunks using FAISS
        Compare query embedding with all chunk embeddings
        Return top-5 most similar chunks
        
Step 5: Return results with context
        (chunk_text, section_name, similarity_score)
```

### API: `search_analysis()`

```python
def search_analysis(
    query: str,           # User's question
    symbol: str,          # Stock symbol (e.g., 'ASIANPAINT')
    top_k: int = 5        # How many results to return
) -> List[Tuple[str, str, float]]:
    """
    Search analysis embeddings for similar content.
    
    Returns:
        [(chunk_text, section_name, similarity_score), ...]
    """
```

**Parameters:**
- `query` (str): The search question or text
- `symbol` (str): Stock to search in (e.g., 'ASIANPAINT')
- `top_k` (int): Number of results (default 5)

**Returns:**
- List of tuples with (chunk_text, section_name, distance_score)
- **Lower distance = more similar** ✅

**Example Return:**
```python
[
    ("Asian Paints has strong ROE of 28%...", "quantitative_analysis", 0.15),
    ("Return on equity demonstrates...", "qualitative_analysis", 0.32),
    ("The ROE metric shows...", "company_overview", 0.45),
]
```

In [8]:
# Example: Searching Embeddings

# Now search the embeddings we just created
test_queries = [
    "What is the ROE?",
    "Tell me about the market share",
    "What is the dividend yield?"
]

print("🔍 Searching the embedding index...\n")
print("=" * 80)

for query in test_queries:
    print(f"\n📝 Query: '{query}'")
    print(f"{'─' * 80}")
    
    # Search the store with custom path
    results = store_with_custom_path.search_analysis(
        query=query,
        symbol='ASIANPAINT',
        top_k=3  # Get top 3 results
    )
    
    if results:
        print(f"Found {len(results)} relevant chunks:")
        for i, (chunk_text, section, distance) in enumerate(results, 1):
            # Lower distance = higher relevance
            relevance_pct = int((1 - min(distance, 1.0)) * 100)
            print(f"\n  {i}. Section: {section}")
            print(f"     Relevance: {relevance_pct}%")
            print(f"     Distance:  {distance:.4f}")
            print(f"     Content:   {chunk_text[:80]}...")
    else:
        print("  ❌ No embeddings found for this stock")

🔍 Searching the embedding index...


📝 Query: 'What is the ROE?'
────────────────────────────────────────────────────────────────────────────────
Found 3 relevant chunks:

  1. Section: shareholding_analysis
     Relevance: 0%
     Distance:  1.7883
     Content:   Promoter holding: 50% controlled by Akzo Nobel N.V. Institutional investors: 30%...

  2. Section: investment_thesis
     Relevance: 0%
     Distance:  1.8193
     Content:   Investment case: Strong brand, market leader, consistent growth. Key triggers: M...

  3. Section: valuation_recommendation
     Relevance: 0%
     Distance:  1.8360
     Content:   Fair value: Rs 3000-3200 based on DCF analysis. Recommendation: BUY for long-ter...

📝 Query: 'Tell me about the market share'
────────────────────────────────────────────────────────────────────────────────
Found 3 relevant chunks:

  1. Section: investment_thesis
     Relevance: 0%
     Distance:  1.0809
     Content:   Investment case: Strong brand, market leader, consist

---

## Section 6️⃣: Loading and Using Embeddings for Q&A

### API: `load_for_qa()`

Used to load embeddings and metadata for building Q&A systems.

```python
def load_for_qa(
    symbol: str  # Stock symbol (e.g., 'ASIANPAINT')
) -> Tuple[Optional[Dict], Optional[List]]:
    """
    Load embeddings and metadata for Q&A.
    
    Args:
        symbol: Stock symbol to load
        
    Returns:
        Tuple of (metadata_dict, chunks_list)
        - metadata_dict: Contains symbol, num_chunks, embedding_model
        - chunks_list: List of (chunk_text, section, symbol) tuples
        
        Returns (None, None) if embeddings not found
    """
```

### What's in the Metadata?

```python
{
    'symbol': 'ASIANPAINT',
    'num_chunks': 45,
    'chunks': [(chunk1, section1, symbol), (chunk2, section2, symbol), ...],
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'
}
```

### Typical Q&A Workflow

```
1. Load embeddings: metadata, chunks = load_for_qa('ASIANPAINT')
2. Get user question: "What is ROE?"
3. Search: results = search_analysis(question, 'ASIANPAINT', top_k=3)
4. Build context from top chunks
5. Send to LLM: "Using this context, answer the question"
6. Return LLM response to user
```

In [9]:
# Example: Loading Embeddings for Q&A

# Load the embeddings we saved
metadata, chunks = store_with_custom_path.load_for_qa('ASIANPAINT')

if metadata and chunks:
    print("✅ Successfully loaded embeddings for Q&A\n")
    
    print("Metadata Information:")
    print(f"  Stock Symbol: {metadata['symbol']}")
    print(f"  Total Chunks: {metadata['num_chunks']}")
    print(f"  Embedding Model: {metadata['embedding_model']}")
    print(f"  Chunks List Length: {len(metadata['chunks'])}\n")
    
    print("Sample Chunks:")
    for i, (chunk_text, section, symbol) in enumerate(chunks[:2], 1):
        print(f"\n  Chunk {i}:")
        print(f"    Section: {section}")
        print(f"    Symbol: {symbol}")
        print(f"    Length: {len(chunk_text)} chars")
        print(f"    Preview: {chunk_text[:80]}...")
else:
    print("❌ Could not load embeddings")

✅ Successfully loaded embeddings for Q&A

Metadata Information:
  Stock Symbol: ASIANPAINT
  Total Chunks: 7
  Embedding Model: sentence-transformers/all-MiniLM-L6-v2
  Chunks List Length: 7

Sample Chunks:

  Chunk 1:
    Section: company_overview
    Symbol: ASIANPAINT
    Length: 341 chars
    Preview: Asian Paints Limited is India's largest paint company. The company has strong pr...

  Chunk 2:
    Section: quantitative_analysis
    Symbol: ASIANPAINT
    Length: 356 chars
    Preview: Financial Performance: Strong revenue growth of 15% CAGR over last 5 years. Prof...


---

## Section 7️⃣: Listing and Managing Stored Analyses

### API: `list_stored_analyses()`

Lists all stocks that have been analyzed and embedded.

```python
def list_stored_analyses() -> List[str]:
    """
    Get list of symbols with stored embeddings.
    
    Returns:
        Sorted list of stock symbols with embeddings
        Example: ['ASIANPAINT', 'EICHERMOT', 'HDFCBANK']
    """
```

**What it does:**
1. Checks the `embeddings/` directory
2. Looks for directories with `faiss_index.bin` file
3. Returns sorted list of symbols

**Storage Structure:**
```
embeddings/
├── ASIANPAINT/
│   ├── faiss_index.bin      (Vector index)
│   └── metadata.pkl         (Chunk metadata)
├── EICHERMOT/
│   ├── faiss_index.bin
│   └── metadata.pkl
└── HDFCBANK/
    ├── faiss_index.bin
    └── metadata.pkl
```

### Full API Reference Table

| Method | Purpose | Returns |
|--------|---------|---------|
| `get_embedding_model()` | Get/load embedding model | EmbeddingModel |
| `get_faiss()` | Get FAISS library | faiss module |
| `chunk_analysis(analysis_result, symbol, chunk_size=500)` | Split analysis into chunks | List[Tuple] |
| `save_analysis_embeddings(symbol, analysis_result, overwrite=True)` | Save to FAISS index | Tuple[bool, str] |
| `search_analysis(query, symbol, top_k=5)` | Search embeddings | List[Tuple] |
| `load_for_qa(symbol)` | Load for Q&A use cases | Tuple[Dict, List] |
| `list_stored_analyses()` | List all indexed stocks | List[str] |

In [10]:
# Example: Listing Stored Analyses

# List all stored analyses in our custom storage
stored = store_with_custom_path.list_stored_analyses()

print("📊 Stored Analyses List\n")
print(f"Total stocks indexed: {len(stored)}")

if stored:
    print("\nStocks with embeddings:")
    for symbol in stored:
        print(f"  ✅ {symbol}")
        
        # Get info about each stock
        metadata, chunks = store_with_custom_path.load_for_qa(symbol)
        if metadata:
            print(f"     → {metadata['num_chunks']} chunks")
            print(f"     → Embedding model: {metadata['embedding_model']}")
else:
    print("\nNo stocks indexed yet")

📊 Stored Analyses List

Total stocks indexed: 1

Stocks with embeddings:
  ✅ ASIANPAINT
     → 7 chunks
     → Embedding model: sentence-transformers/all-MiniLM-L6-v2


---

## Section 8️⃣: Building a Complete Workflow

### End-to-End Example: Stock Analysis Q&A System

Here's how all the pieces work together to build an intelligent stock Q&A system:

```
┌─────────────────────────────────────────────────────────────────┐
│                    USER ASKS A QUESTION                         │
│                  "What is Asian Paints' ROE?"                   │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│  Step 1: Initialize Embedding Store                             │
│  store = AnalysisEmbeddingStore()                               │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│  Step 2: Check if analysis already indexed                      │
│  stored = store.list_stored_analyses()  # ['ASIANPAINT', ...]  │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
        ┌────────────────────┴────────────────────┐
        │                                         │
        ▼                                         ▼
   NOT INDEXED                              ALREADY INDEXED
   (First time)                             (Quick path)
        │                                         │
        ▼                                         ▼
  Step 3a: Analyze stock          Step 3b: Load embeddings
  Run analysis engine             metadata, chunks = 
  Get analysis result dict        load_for_qa('ASIANPAINT')
   (7 sections)                         │
        │                               │
        ▼                           SKIP TO STEP 5
  Step 4: Save embeddings
  save_analysis_embeddings(
    'ASIANPAINT', 
    analysis_result
  )
        │
        └──────────────┬──────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────────┐
│  Step 5: Search for relevant information                        │
│  results = search_analysis(                                     │
│    query="What is Asian Paints' ROE?",                         │
│    symbol='ASIANPAINT',                                        │
│    top_k=3                                                      │
│  )                                                              │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│  Step 6: Get relevant chunks                                    │
│  Results: [                                                     │
│    ("ROE is 28% for ASIANPAINT", "quant_analysis", 0.12),    │
│    ("Return on equity metrics...", "qualitative", 0.25),      │
│    ("ROE trends over 5 years...", "conclusion", 0.35)         │
│  ]                                                              │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│  Step 7: Build context for LLM                                  │
│  context = "\n".join([chunk for chunk, _, _ in results])      │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│  Step 8: Query LLM with context                                 │
│  prompt = f"Based on this: {context}                           │
│            Answer: {question}"                                  │
│  response = llm.invoke(prompt)                                  │
└────────────────────────────┬────────────────────────────────────┘
                             ↓
┌─────────────────────────────────────────────────────────────────┐
│   FINAL ANSWER TO USER                                          │
│   "Based on the analysis, Asian Paints' ROE is 28%..."        │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Complete Workflow Example: Stock Analysis Q&A System

import logging

# Setup logging to see what's happening
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("\n" + "="*80)
print("COMPLETE WORKFLOW: Stock Analysis Q&A System")
print("="*80 + "\n")

# Initialize the store
print("📊 Step 1: Initialize Embedding Store")
store = AnalysisEmbeddingStore(storage_path=Path(temp_storage))
print("  ✅ Store initialized\n")

# Check what's already indexed
print("📊 Step 2: Check Stored Analyses")
existing_stocks = store.list_stored_analyses()
print(f"  Already indexed: {existing_stocks}\n")

# We'll use ASIANPAINT if already indexed, or create it
if 'ASIANPAINT' not in existing_stocks:
    print("📊 Step 3: Saving Analysis (First Time Only)")
    success, msg = store.save_analysis_embeddings('ASIANPAINT', analysis_data, overwrite=False)
    print(f"  {msg}\n")
else:
    print("📊 Step 3: ASIANPAINT Already Indexed (Skipping save)\n")

# Real-world user questions
user_questions = [
    "What is the dividend yield?",
    "How is Asian Paints performing financially?",
    "What are the investment risks?"
]

print("📊 Step 4: Q&A Loop\n")

for question in user_questions:
    print(f"❓ User Question: '{question}'")
    print(f"{'─'*80}")
    
    # Step 5: Search for relevant chunks
    results = store.search_analysis(
        query=question,
        symbol='ASIANPAINT',
        top_k=2  # Get top 2 relevant chunks
    )
    
    if results:
        print(f"Found {len(results)} relevant chunks:\n")
        
        # Step 6: Show what we found
        for i, (chunk_text, section, distance) in enumerate(results, 1):
            relevance = int((1 - min(distance, 1.0)) * 100)
            print(f"  Result {i} (Relevance: {relevance}%):")
            print(f"    Section: {section}")
            print(f"    Chunk: {chunk_text[:120]}...\n")
        
        # Step 7-8: In real system, this would go to LLM
        print("  ✅ In a real system, these chunks would be sent to Claude/GPT")
        print("  ✅ LLM would synthesize a natural answer\n")
    else:
        print("  ❌ No relevant information found\n")
    
    print("-"*80 + "\n")

print("="*80)
print("✅ Workflow Complete!")
print("="*80)

---

## 🎯 Key Takeaways

### What We Learned:

1. **Embeddings are Powerful** 🧠
   - Convert text to numerical vectors (384-dim)
   - Capture semantic meaning, not just keywords
   - Enable intelligent search and Q&A

2. **Chunking is Essential** 📄
   - Break long documents into manageable pieces
   - Use overlapping chunks for context
   - Store metadata for source tracking

3. **FAISS is Fast** ⚡
   - Purpose-built vector database
   - Searches millions of embeddings instantly
   - Uses L2 distance to find similar vectors

4. **Pickles Save State** 💾
   - Binary format for Python objects
   - Perfect for storing chunks and metadata
   - Can be loaded quickly for Q&A workflows

5. **APIs Make it Simple** 🔌
   - `save_analysis_embeddings()` - Store once
   - `search_analysis()` - Find relevant chunks
   - `load_for_qa()` - Load for question answering
   - `list_stored_analyses()` - Manage multiple stocks

### Real-World Architecture:

```
User Question
    ↓
Embedding Store (AnalysisEmbeddingStore)
    ├─ Load embeddings from disk
    ├─ Calculate similarity with FAISS
    ├─ Return top relevant chunks
    ↓
Build Context from Chunks
    ↓
Send to Language Model (Claude/GPT)
    ↓
Natural Language Answer to User
```

### Why This Matters:

✅ **Better Q&A** - Semantic search finds relevant info even with different wording
✅ **Scalable** - Handles thousands of stocks efficiently
✅ **Fast** - FAISS indexes return results in milliseconds
✅ **Maintainable** - Clean separation of concerns (chunk, embed, store, search)
✅ **Real-world Ready** - Used in production Stock Analyzer system

---

## 🔧 Troubleshooting & Common Issues

### Issue 1: "No embeddings found for symbol"
**Cause:** Embedding files not saved yet
```python
# Solution: Save embeddings first
success, msg = store.save_analysis_embeddings('SYMBOL', analysis_dict)
```

### Issue 2: "FAISS not installed"
**Cause:** Missing dependency
```bash
# Solution: Install FAISS
pip install faiss-cpu  # For CPU
# OR
pip install faiss-gpu  # For GPU (NVIDIA)
```

### Issue 3: "Embedding model loading fails"
**Solution:** System has fallback layers
- Try #1: langchain-huggingface
- Try #2: sentence-transformers
- Try #3: langchain-community  
- Try #4: Hash-based (always works!)

### Issue 4: "Search results not relevant"
**Cause:** Query too vague or chunks too long
```python
# Solution: Adjust chunk size and overlap
chunks = store.chunk_analysis(
    analysis_result,
    symbol,
    chunk_size=300  # Smaller chunks
)
```

---

## 🚀 Advanced Topics for the Curious

### 1. Different Index Types
The current implementation uses **IndexFlatL2** (exact, slow but accurate).
For larger datasets (1M+ embeddings), consider:
- **IndexIVF** - Faster, approximate
- **IndexHNSW** - Graph-based, very fast
- **IndexPQ** - Product quantization, compressed

### 2. Hybrid Search
Combine multiple scoring methods:
```
Final Score = Vector Similarity (40%) + Keyword Match (60%)
```
Great for handling multiple stock queries!

### 3. Metadata Filtering
Filter chunks by section before search:
```python
def search_by_section(query, symbol, section='quantitative_analysis'):
    results = search_analysis(query, symbol, top_k=10)
    return [r for r in results if r[1] == section]
```

### 4. Batch Processing
Process multiple stocks efficiently:
```python
stocks = ['ASIANPAINT', 'EICHERMOT', 'HDFCBANK']
for stock in stocks:
    store.save_analysis_embeddings(stock, analysis_data[stock])
```

### 5. Monitor Embedding Quality
Check similarity scores to validate quality:
```python
test_queries = {
    'roe': 'What is ROE?',
    'dividend': 'Dividend yield?',
    'price': 'Stock price?'
}

for topic, query in test_queries.items():
    results = search_analysis(query, 'ASIANPAINT', top_k=1)
    print(f"{topic}: {results[0][2]:.2f}")  # distances
```

---

## 📚 Complete API Reference

### AnalysisEmbeddingStore Class

#### Initialization
```python
from src.embeddings.analysis_embedding_store import AnalysisEmbeddingStore
from pathlib import Path

# Default storage: embeddings/
store = AnalysisEmbeddingStore()

# Custom storage location
store = AnalysisEmbeddingStore(storage_path=Path('/custom/path'))
```

#### Methods

| Method | Purpose | Parameters | Returns |
|--------|---------|------------|---------|
| `chunk_analysis()` | Split text into chunks | `analysis_result: Dict, symbol: str, chunk_size: int = 500` | `List[Tuple[str, str, str]]` |
| `save_analysis_embeddings()` | Save to FAISS index | `symbol: str, analysis_result: Dict, overwrite: bool = True` | `Tuple[bool, str]` |
| `search_analysis()` | Query embeddings | `query: str, symbol: str, top_k: int = 5` | `List[Tuple[str, str, float]]` |
| `load_for_qa()` | Load for Q&A | `symbol: str` | `Tuple[Optional[Dict], Optional[List]]` |
| `list_stored_analyses()` | List indexed stocks | None | `List[str]` |

### Helper Functions

```python
from src.embeddings.analysis_embedding_store import get_embedding_model, get_faiss

# Get embedding model with fallbacks
model = get_embedding_model()
embeddings = model.embed_query("Your text here")
batch_embeddings = model.embed_documents(["text1", "text2"])

# Get FAISS library
faiss = get_faiss()  # Returns faiss module or raises ImportError
```

### Data Types

**Analysis Result Dict:**
```python
analysis_result = {
    'company_overview': '<p>HTML content...</p>',
    'quantitative_analysis': '<p>HTML content...</p>',
    'qualitative_analysis': '<p>HTML content...</p>',
    'shareholding_analysis': '<p>HTML content...</p>',
    'investment_thesis': '<p>HTML content...</p>',
    'valuation_recommendation': '<p>HTML content...</p>',
    'conclusion': '<p>HTML content...</p>'
}
```

**Search Results (Tuple):**
```python
(
    chunk_text: str,        # The text content
    section_name: str,      # 'company_overview', 'quantitative_analysis', etc.
    similarity_score: float # Distance from query (lower = more similar)
)
```

**Metadata Dict:**
```python
{
    'symbol': 'ASIANPAINT',
    'num_chunks': 45,
    'chunks': [(text, section, symbol), ...],
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'
}
```

---

## 📖 Learning Resources

### From This Project:
- **Code Location:** `src/embeddings/analysis_embedding_store.py`
- **Tests:** Check test files for more examples
- **Logs:** `debug_prompts/` and `llm_responses/` directories

### External Resources:
- **FAISS Documentation:** https://github.com/facebookresearch/faiss
- **Sentence Transformers:** https://www.sbert.net/
- **Embeddings Concept:** https://huggingface.co/course/chapter3/2/
- **Vector Databases:** https://www.pinecone.io/learn/vector-database/

### Key Concepts Demystified:

**Vector Space:**
- Imagine each word/document as a point in 384-dimensional space
- Similar meanings cluster together
- Distance between points = dissimilarity

**L2 Distance (Euclidean):**
- Standard distance formula in geometry
- Works well for embeddings
- √[(x₁-y₁)² + (x₂-y₂)² + ... ]

**Cosine Similarity:**
- Alternative metric (angle between vectors)
- Better than L2 for very high dimensions
- Used in: SearchScore = 1 - cosine_distance

**Attention Mechanism:**
- How transformers "pay attention" to relevant parts
- What makes sentence-transformers understand meaning
- Foundation of modern embeddings

---

## 💡 Challenge Yourself: Quiz & Exercises

### Quick Quiz (Answers below)

**Q1: What is the embedding dimension for all-MiniLM-L6-v2?**
A) 256   B) 384 ✓   C) 512   D) 1024

**Q2: In semantic search, what does lower distance score mean?**
A) Less relevant   B) More relevant ✓   C) Older data   D) Bigger chunk

**Q3: What is the chunk overlap size in the implementation?**
A) 50 chars   B) 100 chars ✓   C) 200 chars   D) 500 chars

**Q4: Which method saves embeddings to FAISS?**
A) search_analysis()
B) load_for_qa()
C) save_analysis_embeddings() ✓
D) list_stored_analyses()

**Q5: What format are chunks stored in?**
A) JSON   B) Pickle ✓   C) CSV   D) XML

### Hands-On Challenges

**Challenge 1: Custom Chunk Size**
Modify the chunking to use 300-char chunks instead of 500:
```python
chunks = store.chunk_analysis(analysis_data, 'SYMBOL', chunk_size=300)
print(f"Created {len(chunks)} chunks with size 300")
```

**Challenge 2: Top-K Sensitivity**
Try different top_k values and observe how results change:
```python
for k in [1, 3, 5, 10]:
    results = store.search_analysis(query, 'ASIANPAINT', top_k=k)
    print(f"top_k={k}: {len(results)} results")
```

**Challenge 3: Query Variations**
Same question, different wording - compare results:
```python
queries = [
    "What is ROE?",
    "Return on equity?",
    "How is the stock performing on equity returns?"
]

for q in queries:
    results = store.search_analysis(q, 'ASIANPAINT', top_k=1)
    print(f"Query: {q}")
    print(f"Top result distance: {results[0][2]:.4f}\n")
```

**Challenge 4: Multi-Stock Analysis**
Index multiple stocks and perform cross-stock search:
```python
# Add more stocks
for stock in ['EICHERMOT', 'HDFCBANK']:
    store.save_analysis_embeddings(stock, analysis_dict[stock])

# List all
all_stocks = store.list_stored_analyses()
print(f"Indexed: {all_stocks}")
```

**Challenge 5: Metadata Inspection**
Deep dive into metadata structure:
```python
metadata, chunks = store.load_for_qa('ASIANPAINT')

# Analyze chunk distribution
sections = {}
for text, section, symbol in chunks:
    sections[section] = sections.get(section, 0) + 1

print("Chunks per section:")
for section, count in sorted(sections.items()):
    print(f"  {section}: {count}")
```

---

## 🎓 Next Steps to Master This

1. **Run the Examples** - Execute each code cell in this notebook
2. **Modify Parameters** - Change chunk sizes, top_k, chunk_size
3. **Add More Stocks** - Index different stocks
4. **Build Q&A** - Connect to an LLM (Claude/GPT)
5. **Deploy** - Put in production with proper error handling
6. **Fine-tune** - Adjust embedding model if needed
7. **Monitor** - Track search quality metrics

### Your Learning Journey ✨

```
Beginner: Understand what embeddings are
    ↓
Intermediate: Use AnalysisEmbeddingStore APIs
    ↓
Advanced: Optimize search quality and performance
    ↓
Expert: Build production systems with embeddings
    ↓
Master: Design multi-modal embedding strategies
```

---

## 🏆 Summary

You now understand:
- ✅ How embeddings capture semantic meaning
- ✅ Why chunking improves search quality
- ✅ How FAISS enables lightning-fast vector search
- ✅ How to implement semantic Q&A systems
- ✅ The complete workflow from analysis to answer

**You're ready to build AI-powered systems!** 🚀

Happy coding! 💻